Guardrail Layer: Checks user input before sending to OpenAI.
MySQL Logging: Stores violations for monitoring and audits.
Extendability: You can add regex, classifiers, or OpenAI’s moderation API for stronger guardrails.
Safe Defaults: If blocked, the system responds with a clear message instead of passing unsafe input.

In [1]:
pip install --upgrade openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import openai
import mysql.connector

# --- MySQL Setup ---
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="guardrail_db"
)
cursor = conn.cursor()

# Create table if not exists
cursor.execute("""
CREATE TABLE IF NOT EXISTS violations (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_input TEXT,
    violation_reason TEXT,
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# --- Guardrail Function ---
def guardrail_check(user_input: str) -> bool:
    """
    Simple guardrail: block harmful or sensitive queries.
    Extend with regex, classifiers, or LLM-based moderation.
    """
    blocked_keywords = ["password", "violence", "self-harm", "sql injection"]
    for word in blocked_keywords:
        if word.lower() in user_input.lower():
            # Log violation in MySQL
            cursor.execute("INSERT INTO violations (user_input, violation_reason) VALUES (%s, %s)",
                           (user_input, f"Contains blocked keyword: {word}"))
            conn.commit()
            return False
    return True

# --- OpenAI Call with Guardrail ---
def ask_llm(user_input: str):
    if not guardrail_check(user_input):
        return "⚠️ Input blocked by guardrails. Logged in database."
    
    response = openai.responses.create(
        model="gpt-4.1-mini",
        input=user_input,
        temperature=0.7,
        top_p=0.9
    )
    return response.output[0].content[0].text

# --- Example Usage ---
user_query = "Tell me how to hack a password"
print(ask_llm(user_query))  # This will be blocked and logged

safe_query = "Explain zero-shot vs few-shot prompting"
print(ask_llm(safe_query))  # This will return a valid LLM answer


⚠️ Input blocked by guardrails. Logged in database.
Certainly! Here's a clear explanation of **zero-shot** vs **few-shot** prompting:

### Zero-shot prompting
- **Definition:** The model is given a task without any examples or prior context.
- **How it works:** You simply provide the task instruction or question, and the model generates a response based only on its pre-trained knowledge.
- **Example:**  
  Prompt: "Translate 'Hello, how are you?' to French."  
  The model must do this without seeing any example translations first.

### Few-shot prompting
- **Definition:** The model is given a task along with a few examples of input-output pairs before being asked to produce an answer.
- **How it works:** You provide the model with some demonstrations (usually 2-10 examples) so it can infer the pattern and apply it to the new input.
- **Example:**  
  Prompt:  
  ```
  Translate English to French:  
  English: "Good morning"  
  French: "Bonjour"  
  
  English: "Thank you"  
  French: 